# [Cell 0] Colab Full-Dataset CFM Training Guide

## Recommended flow (Stage 2 not required)

Run in this order:

| Step | Cell | Purpose | Time |
|------|------|---------|------|
| 1 | **1** | Set project directory | < 1 min |
| 2 | **2** | Install dependencies | 2–5 min |
| 3 | **3** | WandB login (optional) | 1 min |
| 4 | **4** | Check required data files | < 1 min |
| 5 | **8** | Prepare extractor bundle (`best.ckpt` + `args.json`) | < 1 min |
| 6 | **9** | Resolve `extractor_path` | < 1 min |
| 7 | **12** | Stage 3 CFM full training | 45–180 min |
| 8 | **13** | Visualize GT vs inference + metrics | 1–3 min |

Optional debug/smoke cells: **5, 6, 7, 10, 11**.

## CFM condition
`c = [hidden_state, direction, DI]`
- `hidden_state`: from Stage 1 GRU extractor (`best.ckpt`)
- `direction`, `DI`: from `Data/DamagePattern/label.csv`

## Note about Cell 8
If `args.json` is missing, Cell 8 reads `hyper_parameters` directly from `best.ckpt` and writes a compatible `args.json`.

## Inference snapshots during training
CFM saves/logs inference images every 10 epochs by default:
- `results/.../inference_samples/epoch_XXXXX.png`


In [ ]:
# [Cell 1] Set project directory
PROJECT_DIR = "/content/intership-project"

import os
assert os.path.isdir(PROJECT_DIR), f"Project dir not found: {PROJECT_DIR}"
%cd /content/intership-project
print("Project dir:", PROJECT_DIR)


In [ ]:
# [Cell 2] Install dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt


In [ ]:
# [Cell 3] Optional WandB login
USE_WANDB = True  # set False to disable

if USE_WANDB:
    import wandb
    wandb.login()
    WANDB_FLAG = "--use_wandb"
else:
    WANDB_FLAG = ""

print("WANDB_FLAG:", WANDB_FLAG)


In [ ]:
# [Cell 4] Check required data files
required_paths = [
    "Data/Hysteresis/info_w_mander_big.csv",
    "Data/Hysteresis/id_list_small.json",
    "Data/DamagePattern/label.csv",
    "Data/DamagePattern/id_list.json",
]

for p in required_paths:
    print(p, "OK" if os.path.exists(p) else "MISSING")


In [ ]:
# [Cell 5] Optional one-click runner
MODE = "smoke"      # "smoke" or "full"
BACKEND = "cfm"     # "gan" or "cfm"

wandb_opt = "--with-wandb" if USE_WANDB else ""
!bash scripts/colab_quickstart.sh {wandb_opt} --mode={MODE} --backend={BACKEND}


In [ ]:
# [Cell 6] Stage 1 smoke train (optional)
!python train_hysteresis.py --architecture HystereticGRU --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# [Cell 7] Stage 2 smoke train (optional)
!python train_quantification.py --backbone vit_b_16 --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# [Cell 8] Prepare extractor bundle for Stage 3
# - If best.ckpt is in project root, copy it to phase1_extractor/
# - If args.json is missing, generate from ckpt hyper_parameters

import os, json, shutil, torch

PROJECT_ROOT = "/content/intership-project"
EXTRACTOR_DIR = os.path.join(PROJECT_ROOT, "phase1_extractor")
os.makedirs(EXTRACTOR_DIR, exist_ok=True)

ckpt_candidates = [
    os.path.join(EXTRACTOR_DIR, "best.ckpt"),
    os.path.join(PROJECT_ROOT, "best.ckpt"),
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)
assert ckpt_path is not None, "No best.ckpt found (expected in project root or phase1_extractor/)"

target_ckpt = os.path.join(EXTRACTOR_DIR, "best.ckpt")
if os.path.abspath(ckpt_path) != os.path.abspath(target_ckpt):
    shutil.copy2(ckpt_path, target_ckpt)
    ckpt_path = target_ckpt
    print("Copied best.ckpt ->", target_ckpt)
else:
    print("Using best.ckpt ->", ckpt_path)

args_path = os.path.join(EXTRACTOR_DIR, "args.json")
if not os.path.exists(args_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    hp = ckpt.get("hyper_parameters") or {}
    assert hp, "Checkpoint has no hyper_parameters; please provide matching args.json"

    # Safe defaults aligned with train_hysteresis.py and train_GRU.sh
    args_payload = {
        "architecture": "HystereticGRU",
        "hidden_state_dim": 512,
        "layers": 4,
        "MLP_activation": "softsign",
        "bidirectional": True,
        "simulate": True,
        "scale": True,
        "data_size": "big",
        "lr": 1e-4,
        "epoch": 5000,
        "batch_size": 8,
        "num_workers": 2,
        "maximum": 1.0,
        "data_type": {},
        "data_version": 1,
    }
    args_payload.update(hp)

    # Infer signal_feat if missing
    sd = ckpt["state_dict"]
    if "signal_feat" not in args_payload:
        if "L.gru.weight_ih_l0" in sd:
            args_payload["signal_feat"] = int(sd["L.gru.weight_ih_l0"].shape[1])
        elif "L.lstm.weight_ih_l0" in sd:
            args_payload["signal_feat"] = int(sd["L.lstm.weight_ih_l0"].shape[1])
        else:
            raise RuntimeError("Cannot infer signal_feat from checkpoint weights")

    with open(args_path, "w", encoding="utf-8") as f:
        json.dump(args_payload, f, indent=2)
    print("Generated args.json ->", args_path)
else:
    print("Using existing args.json ->", args_path)

print("Extractor bundle ready:", EXTRACTOR_DIR)


In [ ]:
# [Cell 9] Resolve extractor_path
import glob, os

# Preferred manual directory for your imported Phase-1 checkpoint bundle.
# Put both files inside this directory (or its subfolders): args.json + best.ckpt
MANUAL_EXTRACTOR_PATH = "/content/intership-project/phase1_extractor"

# Extra auto-search candidates (first valid one will be used)
AUTO_CANDIDATES = [
    MANUAL_EXTRACTOR_PATH,
    "/content/intership-project",
    "/content/phase1_extractor",
]

# Fallback author path (if exists)
AUTHOR_EXTRACTOR = "results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47"


def latest_run(base_glob):
    runs = sorted(glob.glob(base_glob), key=os.path.getmtime)
    assert runs, f"No runs found for: {base_glob}. Run Cell 6 first."
    return runs[-1]


def has_required_extractor_files(run_dir):
    if not os.path.isdir(run_dir):
        return False
    has_args = len(glob.glob(os.path.join(run_dir, "**", "args.json"), recursive=True)) > 0
    has_ckpt = len(glob.glob(os.path.join(run_dir, "**", "best.ckpt"), recursive=True)) > 0
    return has_args and has_ckpt


extractor_path = None
for cand in AUTO_CANDIDATES:
    if has_required_extractor_files(cand):
        extractor_path = cand
        print("Using manual/auto extractor:", extractor_path)
        break

if extractor_path is None and has_required_extractor_files(AUTHOR_EXTRACTOR):
    extractor_path = AUTHOR_EXTRACTOR
    print("Using author extractor:", extractor_path)

if extractor_path is None:
    extractor_path = latest_run("results/HystereticPrediction/HystereticGRU/*")
    print("Using latest local Stage 1 run:", extractor_path)

assert has_required_extractor_files(extractor_path), (
    "extractor_path must contain BOTH args.json and best.ckpt. "
    "If you only copied best.ckpt, also copy the matching args.json from the same Phase-1 run."
)


In [ ]:
# [Cell 10] Stage 3 GAN smoke (optional)
!python train_mixed.py \
  --architecture HystereticCGAN \
  --loss_type SGAN \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning False \
  --direction_conditioning False \
  --add_DI_loss False \
  --image_size 128 \
  --z_dim 100 \
  --G_h_size 128 \
  --D_h_size 128 \
  --SELU False \
  --spectral_D False \
  --no_batch_norm False \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# [Cell 11] Stage 3 CFM smoke (optional)
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --cfm_model_channels 128 \
  --cfm_num_res_blocks 2 \
  --cfm_cond_dim 512 \
  --cfm_dropout 0.0 \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# [Cell 12] Stage 3 CFM full training (packaged)
import subprocess, shlex

TRAIN_CFG = {
    "architecture": "CFMADM",
    "cfm_solver": "heun",
    "cfm_ode_steps": 50,
    "extractor_path": extractor_path,
    "data_path": "id_list.json",
    "latent_conditioning": True,
    "direction_conditioning": True,
    "include_di_conditioning": True,
    "add_DI_loss": False,
    "image_size": 128,
    "lr_schedule": "warmup_cosine",
    "warmup_epochs": 100,
    "min_lr_ratio": 0.1,
    "val_every_n_epoch": 10,
    "inference_every_n_epochs": 10,
    "inference_num_images": 6,
    "epoch": 1000,
    "batch_size": 4,
    "num_workers": 0,
}

cmd = ["python", "train_mixed.py"]
for k, v in TRAIN_CFG.items():
    cmd += [f"--{k}", str(v)]
if WANDB_FLAG:
    cmd.append(WANDB_FLAG)

print("Training command:\n", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)


In [ ]:
# [Cell 13] Visualize GT vs CFM inference + metrics
import os, glob, json, argparse, importlib
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from src.dataset import PatternLoopDataset


def first_match(pattern):
    matches = glob.glob(pattern, recursive=True)
    assert matches, f"No file found: {pattern}"
    return matches[0]


def latest_cfm_run(base_glob="results/DamagePatternGeneration/*/*"):
    run_dirs = sorted(glob.glob(base_glob), key=os.path.getmtime, reverse=True)
    for run_dir in run_dirs:
        args_candidates = glob.glob(os.path.join(run_dir, "**", "args.json"), recursive=True)
        if not args_candidates:
            continue
        args_data = json.load(open(args_candidates[0], "r"))
        if args_data.get("architecture") == "CFMADM":
            return run_dir
    raise RuntimeError("No CFM run found under results/DamagePatternGeneration/*/*")


# 1) Resolve latest CFM run folder
run_dir = latest_cfm_run()
print("CFM run_dir:", run_dir)

# 2) Load saved args and checkpoint
args_path = first_match(os.path.join(run_dir, "**", "args.json"))
run_args = argparse.Namespace(**json.load(open(args_path, "r")))

ckpt_candidates = []
for patt in ["best_fid_*.ckpt", "best_ssim_*.ckpt", "*.ckpt"]:
    ckpt_candidates.extend(glob.glob(os.path.join(run_dir, "**", patt), recursive=True))
assert ckpt_candidates, f"No checkpoint found under {run_dir}"
ckpt_path = sorted(set(ckpt_candidates), key=os.path.getmtime)[-1]
print("Using checkpoint:", ckpt_path)

# 3) Load extractor
extractor_args_path = first_match(os.path.join(run_args.extractor_path, "**", "args.json"))
extractor_args = argparse.Namespace(**json.load(open(extractor_args_path, "r")))
extractor = importlib.import_module("src.model.HystereticPrediction").__dict__[extractor_args.architecture](**vars(extractor_args))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
extractor = extractor.to(device)
extractor_ckpt = first_match(os.path.join(run_args.extractor_path, "**", "best.ckpt"))
extractor.load_state_dict(torch.load(extractor_ckpt, map_location=device, weights_only=False)["state_dict"])
extractor.eval()

# 4) Build validation dataset and pick first sample
transform = transforms.Compose([
    transforms.Resize((run_args.image_size, run_args.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

val_ds = PatternLoopDataset(
    root="./Data",
    mode="val",
    transform=transform,
    extractor=extractor,
    extractor_args=extractor_args,
    **vars(run_args),
)

sample_idx = 0
real_data, hidden_state, direction, exp, timestep, DI, info = val_ds[sample_idx]

real_data = real_data.unsqueeze(0).to(device)
hidden_state = hidden_state.unsqueeze(0).to(device)
direction = direction.unsqueeze(0).to(device)
DI = DI.unsqueeze(0).to(device)
info = info.unsqueeze(0).to(device)

# 5) Load CFM model and sample
model = importlib.import_module("src.model.ImageGeneration").__dict__[run_args.architecture](regressor=None, **vars(run_args))
model = model.to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False)["state_dict"], strict=False)
model.eval()

with torch.no_grad():
    c = model.process_condition(info, hidden_state, direction, DI)
    pred = model.sample_with_ode(c, num_steps=50)

# Convert to [0, 1] for metrics and visualization
real_eval = torch.clamp(real_data * 0.5 + 0.5, 0.0, 1.0)
pred_eval = torch.clamp(pred * 0.5 + 0.5, 0.0, 1.0)

# 6) Metrics: PSNR / SSIM / FID
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
fid_metric = FrechetInceptionDistance(feature=2048, reset_real_features=False, normalize=True).to(device)

with torch.no_grad():
    psnr_val = psnr_metric(pred_eval, real_eval).item()
    ssim_val = ssim_metric(pred_eval, real_eval).item()
    fid_metric.update(real_eval, real=True)
    fid_metric.update(pred_eval, real=False)
    fid_val = fid_metric.compute().item()

print(f"PSNR: {psnr_val:.4f}")
print(f"SSIM: {ssim_val:.4f}")
print(f"FID : {fid_val:.4f}")

real_vis = real_eval[0].permute(1, 2, 0).cpu().numpy()
pred_vis = pred_eval[0].permute(1, 2, 0).cpu().numpy()

# 7) Plot GT vs inference + metrics
fig = plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(real_vis)
plt.title("GT")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(pred_vis)
plt.title("CFM Inference")
plt.axis("off")

plt.suptitle(
    f"Exp: {exp} | timestep: {timestep} | val_idx: {sample_idx} | "
    f"PSNR={psnr_val:.2f}, SSIM={ssim_val:.4f}, FID={fid_val:.2f}"
)
plt.tight_layout()
plt.show()

# 8) Save side-by-side figure
save_path = os.path.join(run_dir, "gt_vs_inference.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved figure to:", save_path)